# Preprocessing

### Imports

In [1]:
import json
from datasets import Dataset
from datasets import load_from_disk
import pathlib

print(f"Python: {__import__('sys').version}")
print(f"datasets: {Dataset.__module__.split('.')[0]} - {__import__('datasets').__version__}")

c:\Users\yujie.lim\OneDrive - SK Jewellery\VSCode Projects\Helpful\Data_Ingest\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
datasets: datasets - 5.0.1


### Create dataset

In [9]:
import json
from datasets import Dataset

ANNOTATIONS = "structured_output/output.json"
OUTPUT = "data/raw"


# Load annotations
with open(ANNOTATIONS, "r", encoding="utf-8") as f:
    annotations = json.load(f)


# Convert dictionary → list of rows
samples = []

for filename, data in annotations.items():

    samples.append({
        "image": data["image_path"],
        "fields": data["fields"]
    })


# Create Hugging Face Dataset
dataset = Dataset.from_list(samples)

print(dataset)

# Save
dataset.save_to_disk(OUTPUT)

print(f"Saved dataset to {OUTPUT}")

import json
import pathlib

IMAGE_PATH = pathlib.Path("images/")
JSON_PATH = pathlib.Path("structured_output/output.json")
OUTPUT_PATH = pathlib.Path("images_renamed/")
OUTPUT_PATH.mkdir(exist_ok=True)

# Load JSON
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

new_data = {}

for i, (old_name, item) in enumerate(data.items(), start=1):

    # Get extension from original filename
    suffix = pathlib.Path(old_name).suffix.lower()

    # New filename
    new_name = f"image{i}{suffix}"

    # Old and new image paths
    old_path = IMAGE_PATH / old_name
    new_path = OUTPUT_PATH / new_name

    # Rename physical image
    if old_path.exists():
        old_path.rename(new_path)
    else:
        print(f"WARNING: Image not found: {old_path}")

    # Update image_path
    item["image_path"] = str(new_path.resolve())

    # Add under new filename
    new_data[new_name] = item


# Save updated JSON
with open(JSON_PATH, "w") as f:
    json.dump(new_data, f, indent=4)

print("Done!")



Dataset({
    features: ['image', 'fields'],
    num_rows: 94
})


Saving the dataset (1/1 shards): 100%|██████████| 94/94 [00:00<00:00, 13033.54 examples/s]

Saved dataset to data/raw
Done!


### Train Test Split

In [10]:
INPUT = "data/raw"

TRAIN_OUTPUT = "data/train"
TEST_OUTPUT = "data/test"


# Load dataset
dataset = load_from_disk(INPUT)

print("Original dataset:")
print(dataset)


# Split
split = dataset.train_test_split(
    test_size=0.2,
    seed=42
)


train_dataset = split["train"]
test_dataset = split["test"]


print("\nTrain:")
print(train_dataset)

print("\nTest:")
print(test_dataset)


# Save
train_dataset.save_to_disk(TRAIN_OUTPUT)
test_dataset.save_to_disk(TEST_OUTPUT)

print("\nSaved train/test datasets.")




Original dataset:
Dataset({
    features: ['image', 'fields'],
    num_rows: 94
})

Train:
Dataset({
    features: ['image', 'fields'],
    num_rows: 75
})

Test:
Dataset({
    features: ['image', 'fields'],
    num_rows: 19
})


Saving the dataset (1/1 shards): 100%|██████████| 19/19 [00:00<00:00, 4747.51 examples/s]


Saved train/test datasets.
